### `solver_testing.ipynb` 
*Created: Sept 21, 2026* <br/>
This notebook contains some utilities for testing the convergence and accuracy of some ODE solvers by comparing them with the algorithms in `OrdinaryDiffEq.jl` and with exact solutions, when an exact solution is available.

In [42]:
using OrdinaryDiffEq, CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics
import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  

In [44]:
struct ODETestProblem{F, U, P, S}
    f::F
    u0::U
    tspan::Tuple{Float64, Float64}
    p::P
    exact_solution::S
end 

function ODETestProblem(f::F, u0::U, tspan::NTuple{2,<:Real}, p::P = nothing; exact_solution::S = nothing) where {F,U,P,S}
    tspan = Float64.(tspan)
    return ODETestProblem(f, u0, tspan, p, exact_solution)
end

ODETestProblem

In [2]:
#Import solvers
@nbinclude("explicit_runge_kutta/euler.ipynb")
@nbinclude("explicit_runge_kutta/rk4.ipynb")

#Import ODE test problems
@nbinclude("../ode_library.ipynb");

In [36]:
function compare_solutions(prob::ODETestProblem, reference_alg::OrdinaryDiffEqAlgorithm, custom_alg::A; dt::Real = 0.01) where {A}
    """    
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg :: algorithm that I wrote, that I'm testing 
  
    Valid options for `custom_alg`: 
        - euler
        - rk4

        ...will add more later....
    """
  
    @unpack f, u0, tspan, p = prob

    #STEP 1: Solve the ODE using the custom algorithm 
    custom_sol = custom_alg(f, u0, tspan, p; dt = dt)

    #STEP 2: Solve the ODE using the reference algorithm (an algorithm from OrdinaryDiffEq.jl)
    reference_sol = solve(ODEProblem(f, u0, tspan, p), reference_alg; adaptive = false, dt = dt)

    #Ensure that each algorithm is solving the problem at the time values (a very small tolerance is permitted)
    t_diff = maximum(custom_sol.t .- reference_sol.t)
    t_tol = 1e-10
    
    if t_diff > t_tol
        throw(error("custom_sol.t and reference_sol.t differ by more than " * @sprintf("%.4e", t_tol)))
    end 
    
    #STEP 3: Compute difference between reference solution and custom solution 
    u_custom = custom_sol.u
    u_reference = reference_sol.u
    max_l2_error = maximum(norm.(u_reference .- u_custom))

    return (reference_sol = reference_sol, custom_sol = custom_sol, max_l2_error = max_l2_error)     
end 

compare_solutions (generic function with 1 method)

In [10]:
exp_growth = ODETestProblem(exponential_rhs, 1.0, (0.0, 10.0));
sinusoid = ODETestProblem(sinusoid_rhs, 0.0, (0.0, 10.0));
damped_oscillator = ODETestProblem(damped_oscillator_rhs, [2.0, 0.0], (0.0, 10.0));
lotka_volterra = ODETestProblem(lotka_volterra_rhs, [1.0, 1.0], (0.0, 10.0), (α = 1.5, β = 1.0, δ = 1.0, γ = 3.0));
lorenz_system = ODETestProblem(lorenz_rhs, [1.0, 1.0, 1.0], (0.0, 10.0), (σ = 10.0, ρ = 28.0, β = 8/3));

In [38]:
@testset "Euler" begin

    test_problems = [exp_growth, sinusoid, damped_oscillator, lotka_volterra, lorenz_system]
    
    for prob in test_problems 
        @test compare_solutions(prob, Euler(), euler; dt = 0.01).max_l2_error ≤ 1e-8
    end 
end;

Test Summary: | Pass  Total  Time
Euler         |    5      5  0.2s


In [ ]:
#Next, 